In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 14. The Mathematics of Attention Mechanisms — Query·Key·Value ⭐

> **Learning Objectives**
> - Explain the meaning of Q, K, V from the perspective of information retrieval (database lookup)
> - Derive why Scaled Dot-Product Attention divides by $\sqrt{d_k}$
> - Analyze the time/space complexity of attention: $O(n^2 d)$
> - Directly compare attention operations on CPU vs GPU

## 14.1 Background of Attention Emergence

Bottleneck of RNN: All encoder information compressed into a single final hidden state. Information loss in long sequences.

Bahdanau Attention (2014): Decoder references **all** encoder hidden states at each step.

**Self-Attention** (Transformer): Every token in the same sequence attends to every other token. Learns global dependencies without RNN.

## 14.2 Query·Key·Value (Q, K, V) — Information Retrieval Perspective

Analogous to database search:
- **Query (Q)**: What I am looking for
- **Key (K)**: The index of each item (what this item pertains to)
- **Value (V)**: The actual content of each item

Attention: Compute similarity between Q and all K → Weighted sum of V using similarity scores.

Formula:
$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

- $Q \in \mathbb{R}^{n \times d_k}$: $n$ queries
- $K \in \mathbb{R}^{n \times d_k}$: $n$ keys
- $V \in \mathbb{R}^{n \times d_v}$: $n$ values
- $QK^\top \in \mathbb{R}^{n \times n}$: Scores for each query-key pair

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Scaled Dot-Product Attention implementation (NumPy)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention.
    Q, K, V: (n, d) or (batch, n, d)
    """
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)  # (..., n, n)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

# Without scaling
np.random.seed(0)
n, d = 4, 8
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

output, weights = attention(Q, K, V)
print(f"Q shape: {Q.shape}")
print(f"Attention output: {output.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\nRow sums (softmax check): {weights.sum(axis=-1).round(4)}")


## 14.3 Why Divide by $\sqrt{d_k}$?

Each element of $QK^\top$ is $\sum_{i=1}^{d_k} Q_i K_i$. If $Q, K \sim \mathcal{N}(0, 1)$:

$$\mathrm{Var}(QK^\top_{ij}) = \sum_{i=1}^{d_k} \mathrm{Var}(Q_i K_i) = d_k$$

Thus, the standard deviation of the scores is $\sqrt{d_k}$. When $d_k$ is large, the scores become large, causing softmax to **saturate** (one value near 1, others near 0).

Dividing by $\sqrt{d_k}$ normalizes the variance to 1, allowing softmax to maintain a smooth distribution and enabling better gradient flow.

In [ ]:
# Experiment: effect of sqrt(d_k) scaling
np.random.seed(0)
dks = [8, 64, 512, 2048]
fig, axes = plt.subplots(1, len(dks), figsize=(16, 4))

for ax, d_k in zip(axes, dks):
    n = 4
    Q = np.random.randn(n, d_k)
    K = np.random.randn(n, d_k)
    # Without scaling
    scores_no = Q @ K.T
    # Without scaling
    scores_scaled = scores_no / np.sqrt(d_k)

    # softmax
    p_no = softmax(scores_no)
    p_scaled = softmax(scores_scaled)

    ax.bar(['q0', 'q1', 'q2', 'q3'], p_no[0], alpha=0.5, label='Scaling X', color='red')
    ax.bar(['q0', 'q1', 'q2', 'q3'], p_scaled[0], alpha=0.5, label='Scaling O', color='blue')
    ax.set_title(f'd_k={d_k}\nVariance: {scores_no.var():.1f} → {scores_scaled.var():.1f}')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch14_scaling_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> As d_k grows, without scaling the softmax concentrates on one value (saturation).")


## 14.4 Time and Space Complexity of Attention

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V$$

- $QK^\top$: $n \times d \times n = O(n^2 d)$ operations
- $\mathrm{softmax}(A) V$: $n \times n \times d = O(n^2 d)$
- **Total time**: $O(n^2 d)$
- **Space**: $A \in \mathbb{R}^{n \times n}$ → $O(n^2)$

Problem: As sequence length $n$ grows, $n^2$ explodes. For 8K context, $n^2 = 6.4 \times 10^7$.
→ Solutions: Flash Attention, sparse attention, etc. (see Chapter 27).

In [ ]:
# Visualize attention by sequence length
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
n2 = [n**2 for n in seq_lens]
n2d = [n**2 * 64 for n in seq_lens]  # d=64

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(seq_lens)), n2, alpha=0.7, label='$n^2$ (Space)')
ax.bar(range(len(seq_lens)), [n/1000 for n in n2d], alpha=0.7, label='$n^2 \\cdot d$ / 1000 (Operation)')
ax.set_xticks(range(len(seq_lens)))
ax.set_xticklabels([str(n) for n in seq_lens])
ax.set_xlabel('Sequence Length n')
ax.set_ylabel('Complexity')
ax.set_title('Attention Complexity: $O(n^2 d)$ by Sequence Length')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/ch14_complexity.png', dpi=100, bbox_inches='tight')
plt.show()
print("An 8K context has 64M attention scores. This is why Flash Attention is essential.")


## 14.5 Causal Masking

Autoregressive models like GPT must not look ahead to future tokens.

The mask matrix $M$:
$$M_{ij} = \begin{cases} 0 & \text{if } i \geq j \text{ (past/current)} \\ -\infty & \text{if } i < j \text{ (future)} \end{cases}$$

Applying softmax to $A + M$ results in zero attention weights for future positions.

In [ ]:
# Without scaling
n = 6
mask = np.triu(np.ones((n, n)) * -1e9, k=1)  # upper-triangular -inf
print("Causal Mask (upper-triangular -inf):")
print(mask[:4, :4])

# Without scaling
np.random.seed(0)
scores = np.random.randn(n, n)  # attention scores
print(f"\nOriginal Scores (first row): {scores[0].round(2)}")
masked = scores + mask
print(f"Mask Application (first row): {masked[0].round(2)}")
weights = softmax(masked)
print(f"\nAfter softmax (first row): {weights[0].round(4)}")
print(f"  → the first token can only attend to itself (1.0)")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(scores, cmap='viridis'); plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Original Attention Scores')
im1 = axes[1].imshow(mask != -1e9, cmap='binary'); plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Causal Mask (white=allowed)')
im2 = axes[2].imshow(weights, cmap='Blues'); plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Weights after Mask Application')
plt.tight_layout()
plt.savefig('../figures/ch14_causal_mask.png', dpi=100, bbox_inches='tight')
plt.show()


## 14.6 [CPU/GPU Benchmark ⑤] Attention Time by Sequence Length

When the sequence length $n$ doubles, attention time increases by 4x (memory also increases by 4x).
Let's directly experience the difference between CPU and GPU.

In [ ]:
# Attention Benchmark
import time
from llm_math.bench import time_fn, format_results_table

def attention_torch(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# Sequence lengths
print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for n in [128, 256, 512, 1024, 2048]:
    d = 64
    Q = torch.randn(n, d)
    K = torch.randn(n, d)
    V = torch.randn(n, d)
    res_cpu = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        Q_g, K_g, V_g = Q.cuda(), K.cuda(), V.cuda()
        res_gpu = time_fn(attention_torch, Q_g, K_g, V_g, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ No GPU. Around n=2048, CPU latency becomes noticeably slow.")


In [ ]:
# Use PyTorch SDPA (scaled_dot_product_attention)
print("Using the PyTorch SDPA function:")
print("(automatically selects optimized backends such as Flash Attention internally)\n")

n, d = 512, 64
Q = torch.randn(1, n, d)
K = torch.randn(1, n, d)
V = torch.randn(1, n, d)

# Without scaling
out_manual = attention_torch(Q, K, V)

# PyTorch SDPA (F.scaled_dot_product_attention)
out_sdpa = F.scaled_dot_product_attention(Q, K, V)

print(f"Manual implementation vs SDPA maximum error: {(out_manual - out_sdpa).abs().max():.2e}")
print("\n=> Same result. SDPA is faster because it is optimized internally.")

# Speed Comparison
print("\nSpeed Comparison (n=512):")
t_manual = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=5)
def sdpa_call(Q, K, V):
    return F.scaled_dot_product_attention(Q, K, V)
t_sdpa = time_fn(sdpa_call, Q, K, V, device='cpu', warmup=2, repeat=5)
print(f"  Manual implementation: {t_manual['mean_ms']:.3f} ms")
print(f"  PyTorch SDPA: {t_sdpa['mean_ms']:.3f} ms")
print(f"  Speed Improvement: {t_manual['mean_ms'] / t_sdpa['mean_ms']:.2f}x")


## 14.7 Attention Backpropagation (Conceptual)

Complex differentiation involving the Jacobian of softmax. Key insight:

$$\frac{\partial \mathcal{L}}{\partial Q} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial \mathcal{L}}{\partial A} V K^\top + \ldots\right)$$

PyTorch’s `autograd` handles this automatically. Flash Attention computes this backward pass memory-efficiently (Ch 27).

## 14.8 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Attention | $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ | Weighted sum of values by query-key similarity |
| Scaling | $\frac{1}{\sqrt{d_k}}$ | Normalizes variance to 1 |
| Causal Mask | $M_{ij} = -\infty$ if $i < j$ | Blocks future reference |
| Complexity | $O(n^2 d)$ time, $O(n^2)$ space | Quadratic in sequence length |

## Exercises

1. Derive $\mathrm{Var}(QK^\top_{ij}) = d_k$ when elements of Q, K are iid $\mathcal{N}(0,1)$.
2. Compare attention outputs with and without causal masking.
3. Visualize the attention weight matrix and explain why diagonal entries tend to be large.
4. Measure CPU time for sequence lengths 256, 512, 1024, 2048 and verify $O(n^2)$ scaling.
5. Implement causal attention using PyTorch SDPA’s `is_causal=True` option.

> Solutions: `solutions/ch14_solutions.ipynb`